# 01 — Data (Phase 1, rubric 20%)

**Goal of this notebook.** Turn the raw HAM10000 dataset into clean, *leak-free*
train/val/test splits with a reusable `Dataset`/`DataLoader` pipeline, and to
**justify** every data decision — not just dump results.

Runs in **Google Colab on GPU**. Nothing here trains a model; this is purely
data prep whose outputs (splits, transforms, class weights) feed `02_model_train.ipynb`.

**Phase-1 gate (must pass before we move on):**
1. Assert **zero `lesion_id` overlap** across train/val/test and print per-split class distributions.
2. Show **one augmented batch** (image grid) to confirm the transforms look sane.

## 0. Environment

Confirm we're on a GPU runtime (Runtime → Change runtime type → GPU). Phase 1
doesn't need the GPU, but keeping the same runtime avoids surprises later.

In [ ]:
import sys, platform, torch
print('Python :', sys.version.split()[0], '|', platform.system())
print('Torch  :', torch.__version__)
print('CUDA   :', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 1. Config

One place for the knobs Phase 1 cares about. **`CLASSES` is load-bearing:** the
index order here defines what output neuron *i* means, and it must stay identical
in `02_model_train.ipynb` and `api/model.py` or the served labels will be wrong.
We use the canonical 7 HAM10000 diagnosis codes.

In [ ]:
from pathlib import Path

# Canonical class order — DO NOT reorder without updating training + serving.
CLASSES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

IMG_SIZE   = 224          # ImageNet-pretrained backbones expect 224
BATCH_SIZE = 32
SEED       = 42
VAL_FRAC   = 0.15         # fractions are of *lesions*, not images
TEST_FRAC  = 0.15
NUM_WORKERS = 2           # Colab typically gives 2 vCPUs

import random, numpy as np, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Classes:', CLASSES)

## 2. Get the dataset (Kaggle → Colab)

Dataset: `kmader/skin-cancer-mnist-ham10000`. Kaggle's current auth is a single
token string (`KGAT_...`) from **Kaggle → Settings → API → Create New API Token**,
read from the `KAGGLE_API_TOKEN` env var. **Paste your token below** — do *not*
commit it (this repo is public). The cell downloads + unzips into `/content/ham10000`.

The images ship in **two** part folders (`HAM10000_images_part_1/2`); we merge
them into a single `image_id → filepath` lookup so the rest of the notebook
doesn't care which part a file came from.

In [ ]:
import os, glob, subprocess

DATA_DIR = Path('/content/ham10000')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# --- Kaggle credentials (current single-token 'KGAT_...' scheme) --------
# Paste your token here, or set it another way (e.g. os.environ before Run All).
# NEVER commit the real token — this repo is public.
os.environ.setdefault('KAGGLE_API_TOKEN', 'PASTE_YOUR_KGAT_TOKEN_HERE')
assert not os.environ['KAGGLE_API_TOKEN'].startswith('PASTE_'), \
    'Set KAGGLE_API_TOKEN to your KGAT_... token first.'

# --- Download + unzip (idempotent) -------------------------------------
meta = DATA_DIR / 'HAM10000_metadata.csv'
if not meta.exists():
    subprocess.run(['pip', '-q', 'install', '-U', 'kaggle'], check=True)  # -U: needs KGAT-token support
    subprocess.run(['kaggle', 'datasets', 'download', '-d',
                    'kmader/skin-cancer-mnist-ham10000', '-p', str(DATA_DIR)], check=True)
    import zipfile
    for z in DATA_DIR.glob('*.zip'):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(DATA_DIR)
    print('Downloaded + extracted.')
else:
    print('Already present:', meta)

In [ ]:
# Build image_id -> path over BOTH part folders.
image_paths = {Path(p).stem: p for p in glob.glob(str(DATA_DIR / '**' / '*.jpg'), recursive=True)}
print('Images found:', len(image_paths))  # expect 10015
assert len(image_paths) > 0, 'No .jpg images found — check the download/unzip step.'

## 3. Load metadata + surface the imbalance

Each row is one image; `lesion_id` groups images of the *same* physical lesion,
and `dx` is the diagnosis label. HAM10000 is severely imbalanced — `nv`
(benign nevi) dominates at ~67%, while `df`/`vasc` are ~1%. We print this
explicitly because it drives two later decisions: **weighted loss** (Phase 2)
and reporting **macro-F1** rather than raw accuracy (Phase 3).

In [ ]:
import pandas as pd

df = pd.read_csv(meta)
df['path'] = df['image_id'].map(image_paths)
missing = df['path'].isna().sum()
print('Rows:', len(df), '| rows with no image file:', missing)
df = df.dropna(subset=['path']).reset_index(drop=True)
df['label'] = df['dx'].map(CLASS_TO_IDX)
assert df['label'].notna().all(), 'Unexpected dx value not in CLASSES.'
df.head()

In [ ]:
counts = df['dx'].value_counts().reindex(CLASSES)
pct = (counts / counts.sum() * 100).round(1)
summary = pd.DataFrame({'count': counts, 'pct_%': pct})
print(summary)
print(f"\nMost common class = {counts.idxmax()} ({pct.max()}% of images) — the imbalance to beat.")

In [ ]:
import matplotlib.pyplot as plt
ax = counts.plot(kind='bar', color='#4C72B0')
ax.set_title('HAM10000 class distribution (by image)'); ax.set_ylabel('images')
plt.tight_layout(); plt.show()

## 4. Split by `lesion_id`, not by image (avoid leakage)

**Why this matters.** The same lesion appears in multiple images (different
magnifications/angles). If we split by *image*, near-duplicate views of one
lesion can land in both train and test — the model 'remembers' the lesion and
test accuracy is inflated. Splitting by **`lesion_id`** guarantees every image of
a given lesion stays within a single split.

All images of one lesion share the same `dx`, so we collapse to one row per
lesion, **stratify by class** at the lesion level (keeps rare classes present in
every split), then expand back to images. Split sizes are 70/15/15 of *lesions*.

In [ ]:
from sklearn.model_selection import train_test_split

# One row per lesion with its (single) diagnosis.
lesions = df.groupby('lesion_id')['dx'].first().reset_index()

train_les, tmp_les = train_test_split(
    lesions, test_size=VAL_FRAC + TEST_FRAC,
    stratify=lesions['dx'], random_state=SEED)
rel_test = TEST_FRAC / (VAL_FRAC + TEST_FRAC)
val_les, test_les = train_test_split(
    tmp_les, test_size=rel_test, stratify=tmp_les['dx'], random_state=SEED)

split_of = {}
for name, part in [('train', train_les), ('val', val_les), ('test', test_les)]:
    for lid in part['lesion_id']:
        split_of[lid] = name
df['split'] = df['lesion_id'].map(split_of)

train_df = df[df.split == 'train'].reset_index(drop=True)
val_df   = df[df.split == 'val'].reset_index(drop=True)
test_df  = df[df.split == 'test'].reset_index(drop=True)
print({k: len(v) for k, v in [('train', train_df), ('val', val_df), ('test', test_df)]})

In [ ]:
# Per-split class distribution (proportions should be similar across splits).
dist = pd.DataFrame({
    'train': train_df['dx'].value_counts(normalize=True),
    'val':   val_df['dx'].value_counts(normalize=True),
    'test':  test_df['dx'].value_counts(normalize=True),
}).reindex(CLASSES).mul(100).round(1)
print('Per-split class distribution (% of split):')
print(dist)

## 5. Augmentations — justified for dermoscopy

We want augmentations that reflect *real* acquisition variation without
destroying the diagnostic signal (pigment network, color, texture, asymmetry).

| Transform | Why it's appropriate here |
|---|---|
| `RandomResizedCrop(224, scale=(0.8, 1.0))` | Mild scale/crop jitter mimics framing/zoom differences; kept ≥0.8 so we don't crop the lesion out. |
| `RandomHorizontalFlip` + `RandomVerticalFlip` | Dermoscopy has **no canonical orientation**, so both flips are label-preserving and free data. |
| `RandomRotation(±15°)` | Small rotations model camera angle; large rotations add black corners and buy nothing. |
| `ColorJitter(brightness/contrast/saturation=0.1, hue=0.02)` | Models lighting/white-balance drift. Kept **mild** — color is diagnostic (e.g. melanoma color variegation), so aggressive hue/saturation would corrupt the label. |
| `Normalize(ImageNet stats)` | Backbones are ImageNet-pretrained; must match their input statistics. |

**Deliberately avoided:** heavy color/hue shifts, elastic/grid distortion, cutout
over the lesion, and aggressive blur — all erode the fine texture and color cues
a dermatology model relies on. Val/test use only resize + center-crop + normalize
(deterministic), so evaluation reflects true performance.

In [ ]:
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

## 6. Custom `Dataset` + `DataLoader`

Following the Lab 3 Part 2 custom-`Dataset` pattern: hold a dataframe of
(`path`, `label`), load each image lazily in `__getitem__`, apply the split's
transform. Lazy loading keeps memory flat (10k JPEGs won't fit decoded in RAM).

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class HAM10000Dataset(Dataset):
    def __init__(self, frame, transform):
        self.paths  = frame['path'].tolist()
        self.labels = frame['label'].astype(int).tolist()
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return self.transform(img), self.labels[i]

train_ds = HAM10000Dataset(train_df, train_tf)
val_ds   = HAM10000Dataset(val_df,   eval_tf)
test_ds  = HAM10000Dataset(test_df,  eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
print('batches — train:', len(train_loader), 'val:', len(val_loader), 'test:', len(test_loader))

## 7. Class weights (for Phase 2's weighted loss)

Computed from the **train** split only (never val/test — that would leak).
Inverse-frequency weighting `w_c = N / (K · n_c)` up-weights rare classes so the
loss stops being dominated by `nv`. We save these for `02_model_train.ipynb`.

In [ ]:
import torch
train_counts = train_df['label'].value_counts().reindex(range(NUM_CLASSES)).fillna(0).values
N, K = train_counts.sum(), NUM_CLASSES
class_weights = torch.tensor(N / (K * train_counts), dtype=torch.float)
for c, w, n in zip(CLASSES, class_weights.tolist(), train_counts):
    print(f'{c:6s} n={int(n):5d}  weight={w:.3f}')

# Persist splits + weights so training doesn't re-split (reproducibility).
ART = Path('/content/artifacts'); ART.mkdir(exist_ok=True)
df[['image_id', 'lesion_id', 'dx', 'label', 'split', 'path']].to_csv(ART / 'splits.csv', index=False)
torch.save({'class_weights': class_weights, 'classes': CLASSES}, ART / 'class_weights.pt')
print('\nSaved splits.csv + class_weights.pt to', ART)

## ✅ Phase-1 verification gate

Two checks the spec requires before Phase 2.

### Gate 1 — zero `lesion_id` overlap + per-split distributions

In [ ]:
s_tr = set(train_df['lesion_id']); s_va = set(val_df['lesion_id']); s_te = set(test_df['lesion_id'])
assert s_tr.isdisjoint(s_va), 'LEAK: train∩val lesions'
assert s_tr.isdisjoint(s_te), 'LEAK: train∩test lesions'
assert s_va.isdisjoint(s_te), 'LEAK: val∩test lesions'
print('PASS — no lesion_id overlap across splits.')
print(f'lesions: train={len(s_tr)} val={len(s_va)} test={len(s_te)} total={len(s_tr|s_va|s_te)}')
print('\nImages per split:', {'train': len(train_df), 'val': len(val_df), 'test': len(test_df)})
print('\nPer-split class % (from section 4):'); print(dist)

### Gate 2 — one augmented batch looks sane

De-normalize a training batch and render a grid. Expect visibly varied crops/
flips/rotations but lesions still clearly recognizable (augmentation didn't
destroy the signal).

In [ ]:
import torchvision, numpy as np, matplotlib.pyplot as plt

imgs, labels = next(iter(train_loader))
mean = torch.tensor(IMAGENET_MEAN).view(3,1,1); std = torch.tensor(IMAGENET_STD).view(3,1,1)
grid = torchvision.utils.make_grid(imgs[:16], nrow=4)
grid = (grid * std + mean).clamp(0,1).permute(1,2,0).numpy()
plt.figure(figsize=(8,8)); plt.imshow(grid); plt.axis('off')
plt.title('Augmented training batch (16)'); plt.show()
print('labels:', [CLASSES[i] for i in labels[:16].tolist()])

---
**Phase 1 done.** Outputs for Phase 2: `train_loader/val_loader/test_loader`,
`eval_tf` (reused verbatim for serving), `class_weights`, and `artifacts/splits.csv`.
STOP here and confirm the gate output before starting `02_model_train.ipynb`.